# 01 - Retrieval experiments

Short, interactive companion to `eval/run_eval.py`:

1. Compare the four search modes on a few questions from the demo notes.
2. Look at the cosine-similarity gap between answerable and unanswerable questions (it is what the refusal threshold relies on).
3. Benchmark the from-scratch heap top-k against `np.argsort`, `np.argpartition` and FAISS.

Run it with the project's `venv` selected as the kernel. No API key is needed: nothing here calls an LLM.

In [ ]:
import json, logging, time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from notemind.config import PROJECT_ROOT, load_config, setup_logging
from notemind.pipeline import RAGPipeline

config = load_config()
setup_logging("WARNING")
pipeline = RAGPipeline(config)
pipeline.add_folder(config.paths.demo_dir)
stats = pipeline.build_index()
print(f"{stats.num_chunks} chunks from {stats.num_documents} PDFs "
      f"({'cache' if stats.from_cache else f'built in {stats.seconds:.1f}s'})")

## 1. Search modes side by side

Dense search understands paraphrases; BM25 is strong on rare exact terms (names, acronyms). Hybrid fuses both rankings with Reciprocal Rank Fusion.

In [ ]:
questions = [
    "What goes wrong if the step size in gradient descent is set too high?",   # paraphrase
    "Which shortest-path algorithm works with negative edge weights?",         # rare term: Bellman-Ford
    "What problem does k-means++ solve?",                                       # exact term
    "Why do the early layers of a very deep network barely learn?",            # paraphrase of 'vanishing gradient'
]
rows = []
for q in questions:
    row = {"question": q}
    for mode in ["faiss", "bm25", "hybrid"]:
        out = pipeline.retrieve(q, top_k=3, mode=mode)
        row[mode] = ", ".join(r.chunk.citation for r in out.results) or "(none)"
    rows.append(row)
pd.set_option("display.max_colwidth", 80)
pd.DataFrame(rows)

## 2. Similarity gap for refusals

For every eval question, take the best cosine similarity to any chunk. A useful refusal threshold sits between the two distributions; overlap means some questions will be misjudged by the threshold alone and need the LLM's own refusal.

In [ ]:
eval_set = json.loads((PROJECT_ROOT / "eval" / "eval_set.json").read_text(encoding="utf-8"))
best = {True: [], False: []}
for item in eval_set["questions"]:
    out = pipeline.retrieve(item["question"], top_k=1, mode="faiss")
    best[item["answerable"]].append(out.max_similarity)

fig, ax = plt.subplots(figsize=(8, 3.8))
bins = np.linspace(0, 1, 26)
ax.hist(best[True], bins=bins, color="#2a78d6", alpha=0.85, label=f"Answerable (n={len(best[True])})")
ax.hist(best[False], bins=bins, color="#eb6834", alpha=0.85, label=f"Unanswerable (n={len(best[False])})")
ax.axvline(config.retrieval.min_similarity, color="#52514e", linestyle="--", linewidth=1)
ax.set_xlabel("Best cosine similarity to any chunk"); ax.set_ylabel("Questions")
ax.set_title("Where should the refusal threshold go?", loc="left")
ax.legend(frameon=False)
for side in ("top", "right"):
    ax.spines[side].set_visible(False)
plt.show()
print("answerable  min/median:", round(min(best[True]), 3), round(float(np.median(best[True])), 3))
print("unanswerable max/median:", round(max(best[False]), 3), round(float(np.median(best[False])), 3))

## 3. Heap top-k vs sorting vs FAISS

All four methods return the same top-k for unit vectors. The heap is O(n log k) but runs a Python loop, so it is the *reference* implementation, not the fast path: vectorised NumPy and FAISS (C++/SIMD) win by a wide margin in practice. That trade-off is the point of keeping both.

In [ ]:
import faiss
from notemind.heap_topk import top_k

rng = np.random.default_rng(0)
k, dim, timings = 5, 384, []

def best_of(fn, repeats=3):
    best_time = float("inf")
    for _ in range(repeats):
        start = time.perf_counter(); fn(); best_time = min(best_time, time.perf_counter() - start)
    return best_time * 1000

for n in [1_000, 10_000, 50_000]:
    m = rng.standard_normal((n, dim)).astype("float32"); m /= np.linalg.norm(m, axis=1, keepdims=True)
    q = rng.standard_normal(dim).astype("float32"); q /= np.linalg.norm(q)
    index = faiss.IndexFlatIP(dim); index.add(m)
    scores = m @ q
    expected = list(np.argsort(-scores, kind="stable")[:k])
    assert [i for i, _ in top_k(scores, k)] == expected  # same answer, different speed
    timings.append({
        "n": n,
        "heap top-k (selection only)": best_of(lambda: top_k(scores, k)),
        "np.argsort (full sort)": best_of(lambda: np.argsort(-scores)[:k]),
        "np.argpartition": best_of(lambda: np.argpartition(-scores, k)[:k]),
        "FAISS search (incl. dot products)": best_of(lambda: index.search(q[None, :], k)),
    })
pd.DataFrame(timings).set_index("n").round(3)

**Things to note for interviews**

- Asymptotics vs constants: O(n log k) beats O(n log n) on paper, but a Python loop loses to vectorised C. Measure before optimising.
- For millions of vectors, exact search stops scaling; approximate FAISS indexes (IVF, HNSW) trade a little recall for much faster search.
- The eval script (`eval/run_eval.py`) is the authoritative comparison; this notebook is for exploring.